# glassfix: Colab demo (free GPU)

1. **Compare** the built-in methods (fast, deterministic). The recommended style is `cleanglass`: no reflections at all.
2. **Optional, generative:** truly see-through glass using Qwen-Image-Edit-2509 (Apache-2.0), plus
   Dereflection-Any-Image for comparison only (its Stable Diffusion 2.1 base is not MIT/Apache/BSD).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free tier). The generative step needs a GPU; part 1 works on CPU, just slower.
2. Upload the `glassfix` project folder (the one with `config.yaml`, `glassfix/`, `tools/`, `requirements-colab.txt`) to Google Drive.
3. Put photos in Drive organised as `<PHOTOS_DIR>/<car_id>/*.jpg` (one subfolder per car).

In [ ]:
# --- Edit these paths (all inside your Google Drive) ---
CODE_DIR   = "/content/drive/MyDrive/glassfix"          # the uploaded project folder
PHOTOS_DIR = "/content/drive/MyDrive/glassfix_photos"   # <car_id>/*.jpg
OUTPUT_DIR = "/content/drive/MyDrive/glassfix_results"  # reports + outputs are copied here
RUN_NAME   = "colab_compare"
GEN_RUN    = "colab_generative"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")

Copy the code to local disk (much faster than running from Drive) and keep the model-weights cache in Drive so later sessions skip the download.

In [ ]:
import os, shutil
WORK = "/content/glassfix"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(CODE_DIR, WORK, ignore=shutil.ignore_patterns("models", "results", ".venv*", "__pycache__"))
MODELS_CACHE = "/content/drive/MyDrive/glassfix_models"
os.makedirs(MODELS_CACHE, exist_ok=True)
os.symlink(MODELS_CACHE, f"{WORK}/models")
%cd /content/glassfix

In [ ]:
# Colab already has a CUDA build of torch/torchvision; this installs the rest at the tested versions.
!pip install -q -r requirements-colab.txt
!python -m glassfix doctor --download

In [ ]:
# Optional sanity check: synthetic unit tests (reflection removed, cracks preserved).
!python -m pytest tests -q

## Part 1: compare the built-in methods

In [ ]:
!python -m glassfix compare --input "$PHOTOS_DIR" --output /content/results --run-name "$RUN_NAME"
dst = os.path.join(OUTPUT_DIR, RUN_NAME)
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(f"/content/results/{RUN_NAME}", dst)
print("Saved to", dst, "- open report.html in any browser (it is self-contained)")

## Part 2 (optional): generative see-through glass on the GPU

`glassfix crops` exports one glass crop per photo. An editor then rewrites only the crops, and
`glassfix compose` pastes the result back **inside the glass mask only**: the car body, wipers and
background keep their original pixels.

Qwen-Image-Edit-2509 is a 20B model. With 4-bit loading and CPU offload it is expected to fit a 16 GB
T4, at several minutes per photo. If it runs out of memory, use Kaggle (2× T4, more RAM).
**Generative models invent the interior, so check that photos of the same car still match.**

In [ ]:
!python -m glassfix crops --input "$PHOTOS_DIR" --output /content/results --run-name "$GEN_RUN"

In [ ]:
# Qwen-Image-Edit-2509 (Apache-2.0) + Lightning 8-step LoRA (Apache-2.0)
!pip install -q -r tools/requirements-qwen.txt
!python tools/edit_qwen.py /content/results/$GEN_RUN/crops /content/results/$GEN_RUN/edits_qwen

In [ ]:
# COMPARISON ONLY: Dereflection-Any-Image (built on Stable Diffusion 2.1, OpenRAIL++-M: not for delivery).
# It pins old library versions, so it gets its own virtual environment. Skip this cell if not needed.
RUN_DAI = True
if RUN_DAI:
    !git clone -q https://github.com/Abuuu122/Dereflection-Any-Image /content/DAI-repo || true
    !python -m venv /content/dai_env
    !/content/dai_env/bin/pip install -q -r /content/DAI-repo/requirements.txt pyyaml peft
    !/content/dai_env/bin/python tools/edit_dai.py /content/results/$GEN_RUN/crops /content/results/$GEN_RUN/edits_dai --repo /content/DAI-repo

In [ ]:
edits = f"qwen=/content/results/{GEN_RUN}/edits_qwen"
if os.path.isdir(f"/content/results/{GEN_RUN}/edits_dai"):
    edits += f" dai=/content/results/{GEN_RUN}/edits_dai"
!python -m glassfix compose --output /content/results --run-name "$GEN_RUN" --edits $edits --with-styles cleanglass
dst = os.path.join(OUTPUT_DIR, GEN_RUN)
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(f"/content/results/{GEN_RUN}", dst)
print("Saved to", dst)

In [ ]:
# Optional: preview a report inline.
from IPython.display import HTML, display
display(HTML(open(f"/content/results/{RUN_NAME}/report.html", encoding="utf-8").read()))